# Project 1 ANSC 4040 Final Model Tuning and Performance 

This is a notebook for Project 1 that will be used for expanding the same size and tuning the hyperparameters of original model

In [2]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

Original dataset
       │
       ├── 70% TRAIN ───────────────┐
       │                            │
       │                      Hyperparameter
       │                         tuning
       │                            │
       │                            ↓
       │                       Best RF model
       │                            │
       ├── 20% VALIDATION ──────────┤
       │                            ↓
       │                     Model evaluation
       │
       └── 10% TEST ────────────────→ FINAL evaluation

In [3]:
# Load Data Sets 
train_path = "train_data.csv"
val_path = "val_data.csv"
test_path = "test_data.csv"

In [4]:
# Get Animal IDs from each split

train_ids = set()

for chunk in pd.read_csv(
    train_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000
):
    train_ids.update(
        chunk["AnimalId"].dropna().unique()
    )


val_ids = set()

for chunk in pd.read_csv(
    val_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000
):
    val_ids.update(
        chunk["AnimalId"].dropna().unique()
    )


test_ids = set()

for chunk in pd.read_csv(
    test_path,
    usecols=["AnimalId"],
    dtype={"AnimalId": "string"},
    chunksize=100_000
):
    test_ids.update(
        chunk["AnimalId"].dropna().unique()
    )


# Cows represented in ALL THREE splits
shared_cows = sorted(
    train_ids & val_ids & test_ids
)

print(f"Cows in all three splits: {len(shared_cows):,}")

Cows in all three splits: 8,976


In [5]:
#at least 400 cows
if len(shared_cows) < 400:
    raise ValueError(
        f"Only {len(shared_cows)} cows occur in all three splits."
    )

In [6]:
#select cows
selected_cows = set(
    pd.Series(
        shared_cows,
        dtype="string"
    ).sample(
        n=400,
        random_state=42
    ).tolist()
)

print(f"Selected cows: {len(selected_cows)}")

Selected cows: 400


In [7]:
#define function to load selected cows from a CSV file
import pandas as pd

def load_selected_cows(path, selected_cows):
    chunks = []

    for chunk in pd.read_csv(
        path,
        dtype={"AnimalId": "string"},
        chunksize=100_000
    ):
        selected = chunk[chunk["AnimalId"].isin(selected_cows)]

        if len(selected) > 0:
            chunks.append(selected)

    if not chunks:
        raise ValueError(f"No selected cows found in {path}")

    return pd.concat(chunks, ignore_index=True)

In [8]:
#load cows

train_400 = load_selected_cows(
    train_path,
    selected_cows
)

val_400 = load_selected_cows(
    val_path,
    selected_cows
)

test_400 = load_selected_cows(
    test_path,
    selected_cows
)

print("TRAIN")
print(f"Rows: {len(train_400):,}")
print(f"Cows: {train_400['AnimalId'].nunique()}")

print("\nVALIDATION")
print(f"Rows: {len(val_400):,}")
print(f"Cows: {val_400['AnimalId'].nunique()}")

print("\nTEST")
print(f"Rows: {len(test_400):,}")
print(f"Cows: {test_400['AnimalId'].nunique()}")

TRAIN
Rows: 208,815
Cows: 400

VALIDATION
Rows: 59,658
Cows: 400

TEST
Rows: 29,838
Cows: 400


#check 400 in each set 

In [10]:
#create model features and date variables
feature_cols = [
    "Avgmilkflow",
    "EventYear",
    "EventMonth",
    "EventDay",
    "EventDayOfWeek",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking"
]

for data in [train_400, val_400, test_400]:

    event_date = pd.to_datetime(
        data["EventDate"]
    )

    data["EventYear"] = event_date.dt.year
    data["EventMonth"] = event_date.dt.month
    data["EventDay"] = event_date.dt.day
    data["EventDayOfWeek"] = event_date.dt.dayofweek

In [11]:
#create x and y for training validation and test data

X_train = train_400[feature_cols].astype("float32")
y_train = train_400["AnimalId"]

X_val = val_400[feature_cols].astype("float32")
y_val = val_400["AnimalId"]

X_test = test_400[feature_cols].astype("float32")
y_test = test_400["AnimalId"]

# Tune the training data

In [13]:
#6 configuration x 2 folds 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=1
)

param_distributions = {
    "n_estimators": [50, 75, 100],
    "max_depth": [10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="accuracy",
    cv=2,
    random_state=42,
    n_jobs=1,
    verbose=2
)

random_search.fit(X_train, y_train)


Fitting 2 folds for each of 6 candidates, totalling 12 fits
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=100; total time=  54.2s
[CV] END max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=100; total time=  54.2s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=75; total time= 1.2min
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=75; total time= 1.2min
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=100; total time=  32.6s
[CV] END max_depth=20, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=100; total time=  30.6s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=75; total time=  22.9s
[CV] END max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=75; tot

RandomizedSearchCV(cv=2,
                   estimator=RandomForestClassifier(n_jobs=1, random_state=42),
                   n_iter=6, n_jobs=1,
                   param_distributions={'max_depth': [10, 15, 20],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 75, 100]},
                   random_state=42, scoring='accuracy', verbose=2)

In [ ]:
# Inspect the actual best model parameters if the search object is unavailable
if "best_rf" in globals():
    print(best_rf.get_params())
else:
    param_distributions = {
        "n_estimators": [50, 75, 100],
        "max_depth": [10, 15, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    }
    print(param_distributions)

NameError: name 'random_search' is not defined

In [14]:
#evaluate best model on 20% validation set

best_rf = random_search.best_estimator_

y_val_pred = best_rf.predict(X_val)

val_top1 = accuracy_score(
    y_val,
    y_val_pred
)

print(
    f"Validation Top-1 accuracy: "
    f"{val_top1:.2%}"
)

Validation Top-1 accuracy: 4.40%


In [15]:
#calculate top 5 and 10 accuracy for validation set
y_val_proba = best_rf.predict_proba(X_val)

classes = best_rf.classes_

top_10_indices = np.argsort(
    y_val_proba,
    axis=1
)[:, -10:][:, ::-1]

top_10_predictions = classes[top_10_indices]

y_true = np.asarray(y_val)

val_top5 = np.mean(
    np.any(
        top_10_predictions[:, :5]
        == y_true[:, None],
        axis=1
    )
)

val_top10 = np.mean(
    np.any(
        top_10_predictions[:, :10]
        == y_true[:, None],
        axis=1
    )
)

print(f"Validation Top-1:  {val_top1:.2%}")
print(f"Validation Top-5:  {val_top5:.2%}")
print(f"Validation Top-10: {val_top10:.2%}")

Validation Top-1:  4.40%
Validation Top-5:  16.53%
Validation Top-10: 26.90%


# Train the final model

In [14]:
#train and validation data to train model
X_train_final = pd.concat(
    [X_train, X_val],
    axis=0
)

y_train_final = pd.concat(
    [y_train, y_val],
    axis=0
)

print(
    f"Final training rows: "
    f"{len(X_train_final):,}"
)

Final training rows: 268,473


In [16]:
#create final RF using best hyperparameters from random search
final_rf = RandomForestClassifier(
    max_depth=20,
    max_features='log2',
    min_samples_leaf=4,
    min_samples_split=5,
    n_jobs=1,
    random_state=42
)

final_rf.fit(
    X_train_final,
    y_train_final
)

RandomForestClassifier(max_depth=20, max_features='log2', min_samples_leaf=4,
                       min_samples_split=5, n_jobs=1, random_state=42)

# Final Model Performance on test set

In [17]:
#evaluate model on test set
y_test_pred = final_rf.predict(X_test)

test_top1 = accuracy_score(
    y_test,
    y_test_pred
)

print(
    f"FINAL TEST Top-1 accuracy: "
    f"{test_top1:.2%}"
)

FINAL TEST Top-1 accuracy: 7.12%


In [18]:
#top 5 and 10 accuracy for test set
y_test_proba = final_rf.predict_proba(X_test)

classes = final_rf.classes_

top_10_indices = np.argsort(
    y_test_proba,
    axis=1
)[:, -10:][:, ::-1]

top_10_predictions = classes[top_10_indices]

y_true = np.asarray(y_test)

test_top5 = np.mean(
    np.any(
        top_10_predictions[:, :5]
        == y_true[:, None],
        axis=1
    )
)

test_top10 = np.mean(
    np.any(
        top_10_predictions[:, :10]
        == y_true[:, None],
        axis=1
    )
)

print("\n================================")
print("FINAL TEST RESULTS")
print("================================")

print(f"Top-1:  {test_top1:.2%}")
print(f"Top-5:  {test_top5:.2%}")
print(f"Top-10: {test_top10:.2%}")

print("================================")


FINAL TEST RESULTS
Top-1:  7.12%
Top-5:  23.45%
Top-10: 35.78%
